# Uji Akurasi & Latensi — Model Dual-Branch (Router) vs Vanilla vs P2-Static

**Versi diperbaiki (2026-10-08).** Jalankan **berurutan dari atas ke bawah** di kernel yang baru (*Restart & Run All*). Notebook ini sengaja tidak memakai variabel dari sesi lain.

Perbaikan dibanding versi sebelumnya (hasil audit pengukuran):

| # | Masalah lama | Perbaikan |
|---|---|---|
| 1 | Kalibrasi gate berjalan dalam `model.train()`; `no_grad` tidak menghentikan update BatchNorm, sehingga ±92% running stats tertimpa statistik val. Model yang BN-nya berubah inilah yang diekspor ke TensorRT, sedangkan akurasi dihitung dari checkpoint bersih (aktivasi 18.9% vs 32.1%). | Kalibrasi dalam `model.eval()`. Sidik jari (hash) bobot+buffer diperiksa sebelum/sesudah setiap tahap; notebook berhenti bila model berubah. |
| 2 | Logit untuk fit kalibrasi dihitung ulang dengan kode sendiri (bisa beda dari `compute_gate_only`). | Logit diambil dengan rumus yang sama, lalu keputusan gate dicocokkan 1:1 dengan `compute_gate_only` (assert). |
| 3 | `calib_T/calib_b` di-hardcode (1.3483/0.4569), asal-usul tidak jelas. | Di-fit di split **val**, disimpan ke `calib.json` beserta checkpoint & sidik jari model; test tidak pernah dipakai untuk kalibrasi (assert). |
| 4 | Akurasi (EMA) dan export (`ckpt["model"]`) bisa memakai bobot berbeda; ada dua objek model di sesi. | Satu fungsi pemuat, satu sumber bobot (dicetak), satu objek model untuk semua tahap. |
| 5 | Baseline Vanilla/P2 diukur dengan `TRTEngine` yang `cuda.mem_alloc` di setiap panggilan (±0.6 ms ekstra), router dengan buffer pre-alloc. | Semua model memakai engine pre-alloc yang sama, gambar test asli yang sama, diukur bergantian (interleaved) di sesi yang sama. |
| 6 | Verifikasi ONNX membuka file versi lama (`stage1.onnx`), bukan yang dibangun jadi engine (`*v2.onnx`). | Path ONNX/engine hanya didefinisikan sekali; verifikasi memakai file yang sama persis, pada gambar asli. |
| 7 | mAP hanya dari PyTorch; output TensorRT tidak pernah dievaluasi. | mAP dihitung juga dari output TensorRT dengan validator yang sama; keputusan gate TRT vs PyTorch dicocokkan untuk SEMUA gambar test. |
| 8 | Baseline PyTorch 15.127 ms & aktivasi 0.33 di-hardcode dari sesi lain. | Semua angka diukur di sesi ini; tidak ada konstanta hasil dari run lain. |
| 9 | Warm-up hanya memanaskan satu engine Stage2; statistik tanpa p95. | Semua engine dipanaskan; laporan mean, std, p50, p95, p99, n. |
| 10 | Akurasi Vanilla/P2 dihitung di luar notebook (preprocessing `rect`). | Kelima model dievaluasi dengan dataloader, letterbox, conf, IoU, dan NMS yang identik. |
| 11 | Repo di-clone tanpa pin commit. | `CODE_COMMIT` wajib diisi; hash commit, versi library, hash file dicatat di `provenance.json`. |
| 12 | Definisi "objek kecil" di analisis gate = luas ternormalisasi < 0.01 (±64 px). | Memakai definisi COCO (< 32×32 px pada resolusi asli); definisi lama tetap dicetak sebagai pembanding. |

**Cakupan waktu latency (sama untuk semua model):** copy gambar host→device, inferensi TensorRT, (router: baca 1 skalar gate), copy output device→host. **Tidak termasuk:** decode/letterbox gambar dan NMS.

## 0. Setup lingkungan (Kaggle)

In [ ]:
# ===== 0. Setup — clone repo pada commit yang DIPIN =====
# Isi CODE_COMMIT dengan hash commit kode yang dipakai untuk melatih checkpoint yang dievaluasi.
# Kandidat untuk checkpoint KITTI final (versi lama, pra-Opsi C): fcc3c2660 / 1629ff5f0 / 3e7ae3080 (lihat audit).
# JANGAN memakai main/HEAD untuk checkpoint lama: kode terbaru memakai top-k pooling & GroupNorm,
# sehingga checkpoint lama tetap bisa dimuat tetapi keputusan gate-nya berbeda.
REPO_URL = "https://github.com/maugi1547/thesis-Difficulty-Aware-Router"
CODE_COMMIT = "ISI_HASH_COMMIT"
REPO_DIR = "/kaggle/working/thesis-Difficulty-Aware-Router"
assert CODE_COMMIT != "ISI_HASH_COMMIT", "Isi CODE_COMMIT dengan hash commit yang menghasilkan checkpoint."

import os
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch -q --all
!git -C {REPO_DIR} checkout -q {CODE_COMMIT}
%cd {REPO_DIR}
!pip install -q -e .
!pip install -q onnx onnxscript onnxruntime pycocotools tensorrt pycuda

## 1. Konfigurasi (satu-satunya tempat angka/path ditulis)

In [ ]:
import os, json, time, hashlib, subprocess, copy, platform, datetime
from pathlib import Path
import numpy as np
import torch

CFG = dict(
    repo_dir=globals().get("REPO_DIR", os.getcwd()),
    model_yaml="ultralytics/cfg/models/v8/yolov8n-p2-router.yaml",
    router_ckpt="/kaggle/input/models/agungmaugi/yolov8-p2-router-dualbranch/pytorch/default/1/best_modeldualbranch.pt",
    vanilla_ckpt="/kaggle/input/models/agungmaugi/yolov8n-kitti/pytorch/default/1/Skenario1_YOLOv8n_Standar/weights/best.pt",
    p2static_ckpt="/kaggle/input/models/agungmaugi/yolov8n-p2-statis-kitti/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/best.pt",
    data_yaml="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml",
    ckpt_weights_key="ema",          # "ema" (dipakai Ultralytics saat val) atau "model"; dicetak & dipakai di SEMUA tahap
    imgsz=640,
    conf=0.001, iou=0.7, max_det=300, # identik untuk SEMUA model
    calib_split="val",                # fit kalibrasi
    report_split="test",              # angka yang dilaporkan
    expected_counts={"val": 1297, "test": 1000},
    use_existing_calib=False,         # True = pakai calib.json (hanya bila checkpoint & sidik jari cocok)
    device="cuda:0",
    out_dir="/kaggle/working/eval_out",
    n_warmup=50,                      # iterasi pemanasan per engine (dibuang)
    n_reps=3,                         # pengulangan penuh test set untuk latency
    n_onnx_check=5,                   # jumlah gambar asli untuk verifikasi ONNX
    run_pycocotools=True,
    run_gate_analysis=True,
)
CFG.update(globals().get("CFG_OVERRIDE", {}))   # dipakai hanya untuk uji otomatis
assert CFG["calib_split"] != CFG["report_split"], "Kalibrasi tidak boleh di-fit pada split yang dilaporkan."

OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device(CFG["device"] if (CFG["device"] == "cpu" or torch.cuda.is_available()) else "cpu")
torch.manual_seed(0); np.random.seed(0)
torch.backends.cudnn.benchmark = False
print(json.dumps({k: v for k, v in CFG.items()}, indent=1, default=str))

In [ ]:
# ===== Provenance: commit kode, versi library, hash file =====
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def git(*args):
    try:
        return subprocess.check_output(["git", "-C", CFG["repo_dir"], *args], text=True).strip()
    except Exception as e:
        return f"unknown ({type(e).__name__})"

import ultralytics
PROV = dict(
    timestamp=datetime.datetime.now().isoformat(timespec="seconds"),
    git_commit=git("rev-parse", "HEAD"),
    git_dirty_files=len([l for l in git("status", "--porcelain").splitlines() if l.strip()]) if not git("status").startswith("unknown") else None,
    ultralytics_file=ultralytics.__file__,
    versions=dict(python=platform.python_version(), torch=torch.__version__, ultralytics=ultralytics.__version__),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    files={k: sha256_file(CFG[k]) for k in ("router_ckpt", "vanilla_ckpt", "p2static_ckpt", "data_yaml") if os.path.isfile(CFG[k])},
)
for mod in ("onnx", "onnxruntime", "tensorrt", "pycuda"):
    try:
        PROV["versions"][mod] = __import__(mod).__version__
    except Exception:
        PROV["versions"][mod] = None
if "CODE_COMMIT" in globals():
    assert PROV["git_commit"].startswith(CODE_COMMIT), f"Commit aktif {PROV['git_commit']} != CODE_COMMIT {CODE_COMMIT}"
assert ultralytics.__file__.startswith(str(Path(CFG["repo_dir"]).resolve())) or "CFG_OVERRIDE" in globals(), \
    f"ultralytics di-import dari {ultralytics.__file__}, bukan dari repo yang dipin."
print(json.dumps(PROV, indent=1))

## 2. Data (split dipisah tegas: val untuk kalibrasi, test untuk laporan)

In [ ]:
from ultralytics.data.utils import check_det_dataset
from ultralytics.data import YOLODataset
from ultralytics.data.build import build_dataloader

DATA = check_det_dataset(CFG["data_yaml"])
NC, NAMES = DATA["nc"], DATA["names"]

def make_loader(split, batch):
    # augment=False, rect=False -> letterbox persegi imgsz (sama dengan engine TensorRT fixed-shape)
    ds = YOLODataset(img_path=DATA[split], data=DATA, task="detect", imgsz=CFG["imgsz"],
                     augment=False, rect=False, cache=False, batch_size=batch)
    return build_dataloader(ds, batch=batch, workers=2, shuffle=False, rank=-1)

cal_loader = make_loader(CFG["calib_split"], 16)
test_loader = make_loader(CFG["report_split"], 1)    # batch 1 = sama dengan engine TensorRT
N_CAL, N_TEST = len(cal_loader.dataset), len(test_loader.dataset)
print(f"{CFG['calib_split']}: {N_CAL} gambar | {CFG['report_split']}: {N_TEST} gambar")
for split, n in ((CFG["calib_split"], N_CAL), (CFG["report_split"], N_TEST)):
    exp = CFG["expected_counts"].get(split)
    assert exp is None or n == exp, f"Jumlah gambar {split} = {n}, diharapkan {exp}"
assert not (set(cal_loader.dataset.im_files) & set(test_loader.dataset.im_files)), "Ada gambar yang sama di val dan test!"

## 3. Muat model router (SATU objek, mode eval, sidik jari bobot)

In [ ]:
from ultralytics.model_yolov8_router import DualBranchDetectionModel
from ultralytics.cfg import get_cfg
from ultralytics.utils import DEFAULT_CFG

def model_fingerprint(m):
    # Hash seluruh state_dict (bobot + buffer BN). Berubah sedikit saja -> hash berubah.
    h = hashlib.sha256()
    for k, v in sorted(m.state_dict().items()):
        h.update(k.encode()); h.update(v.detach().float().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def find_router(m):
    return next(x for x in m.modules() if "DifficultyAwareRouter" in type(x).__name__)

def set_calibration(router, T, b):
    router.calib_T = float(T); router.calib_b = float(b)

def get_calibration(router):
    return float(getattr(router, "calib_T", 1.0)), float(getattr(router, "calib_b", 0.0))

def load_dualbranch(path, key):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print("Kunci checkpoint:", [k for k in ckpt.keys()] if isinstance(ckpt, dict) else type(ckpt))
    src = ckpt.get(key) if isinstance(ckpt, dict) else None
    if src is None:
        raise KeyError(f"ckpt['{key}'] kosong — pilih kunci lain di CFG['ckpt_weights_key'] secara SADAR.")
    for other in ("ema", "model"):
        if other != key and isinstance(ckpt, dict) and ckpt.get(other) is not None:
            same = all(torch.equal(a.float(), b.float()) for a, b in zip(ckpt[other].state_dict().values(), src.state_dict().values()))
            print(f"⚠️ ckpt juga punya '{other}' — {'identik' if same else 'BERBEDA'} dengan '{key}'. Hanya '{key}' yang dipakai di seluruh notebook.")
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in src.state_dict().items()}
    m = DualBranchDetectionModel(cfg=CFG["model_yaml"], ch=3, nc=NC, verbose=False)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "calib" not in k]
    assert not bad and not unexpected, f"State dict tidak cocok. missing={bad[:5]} unexpected={list(unexpected)[:5]}"
    hyp = (ckpt.get("train_args") or {}) if isinstance(ckpt, dict) else {}
    m.args = get_cfg(DEFAULT_CFG, overrides={k: hyp[k] for k in ("box", "cls", "dfl") if k in hyp})
    m.names = NAMES
    m = m.to(DEVICE).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

model = load_dualbranch(CFG["router_ckpt"], CFG["ckpt_weights_key"])
router = find_router(model)
FP_MODEL = model_fingerprint(model)
PROV["model_fingerprint"] = FP_MODEL
PROV["ckpt_weights_key"] = CFG["ckpt_weights_key"]
print("Sidik jari model:", FP_MODEL[:16], "| stride A:", model.detect_A.stride.tolist(), "| stride B:", model.detect_B.stride.tolist())
print("box/cls/dfl gain:", model.args.box, model.args.cls, model.args.dfl)

def assert_model_unchanged(where):
    fp = model_fingerprint(model)
    assert fp == FP_MODEL, f"❌ Model BERUBAH setelah: {where}. Hentikan dan diagnosis."
    assert not model.training, f"❌ Model tidak dalam mode eval setelah: {where}"
    print(f"✅ Model tidak berubah setelah: {where}")

## 4. Fungsi forward per-stage (dipakai PyTorch, ekspor ONNX, dan TensorRT)

In [ ]:
import torch.nn as nn

class Stage1_BackboneNeckGate(nn.Module):
    # Layer 0-15 + gate (compute_gate_only, sudah termasuk kalibrasi & threshold)
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.pre_router_layers = nn.ModuleList(ml[0:16])
        self.router = ml[16]
        assert "DifficultyAwareRouter" in type(self.router).__name__
    def forward(self, x):
        y = []
        for m in self.pre_router_layers:
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x)
        p3_neck, p2_backbone, p4_neck, p5_sppf = x, y[2], y[12], y[9]
        gate = self.router.compute_gate_only(p3_neck, p2_backbone)
        return gate, p3_neck, p2_backbone, p4_neck, p5_sppf

class Stage2A_WithP2(nn.Module):
    # Layer 16 (expert) + 17-25 + Detect_A; output = prediksi ter-decode y (B, 4+nc, 34000)
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.router, self.branch_a_layers, self.detect_A = ml[16], nn.ModuleList(ml[17:26]), ml[32]
    def forward(self, p3_neck, p2_backbone, p4_neck, p5_sppf):
        p2_out = self.router.compute_expert_only(p3_neck, p2_backbone)
        y = {15: p3_neck, 12: p4_neck, 9: p5_sppf, 16: p2_out}
        x = p2_out
        for i, m in zip(range(17, 26), self.branch_a_layers):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x
        out = self.detect_A([p2_out, y[19], y[22], y[25]])
        return out[0] if isinstance(out, tuple) else out

class Stage2B_NoP2(nn.Module):
    # Layer 26-31 + Detect_B; output y (B, 4+nc, 8400)
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.branch_b_layers, self.detect_B = nn.ModuleList(ml[26:32]), ml[33]
    def forward(self, p3_neck, p4_neck, p5_sppf):
        y = {15: p3_neck, 12: p4_neck, 9: p5_sppf}
        x = p3_neck
        for i, m in zip(range(26, 32), self.branch_b_layers):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x
        out = self.detect_B([p3_neck, y[28], y[31]])
        return out[0] if isinstance(out, tuple) else out

stage1 = Stage1_BackboneNeckGate(model).eval()
stage2a = Stage2A_WithP2(model).eval()
stage2b = Stage2B_NoP2(model).eval()

# Uji kesetaraan stage vs model utuh pada 2 gambar test ASLI
with torch.no_grad():
    for k, batch in enumerate(test_loader):
        if k >= 2: break
        im = batch["img"].to(DEVICE).float() / 255.0
        g, p3, p2, p4, p5 = stage1(im)
        yA_ref, yB_ref = (o[0] for o in model._predict_once_dual(im))
        dA = ((stage2a(p3, p2, p4, p5) - yA_ref).abs().max() / yA_ref.abs().max()).item()
        dB = ((stage2b(p3, p4, p5) - yB_ref).abs().max() / yB_ref.abs().max()).item()
        print(f"[{k}] error relatif Stage2A vs model: {dA:.2e} | Stage2B vs model: {dB:.2e}")
        assert dA < 1e-5 and dB < 1e-5, "Stage tidak setara dengan model utuh"
assert_model_unchanged("uji kesetaraan stage")

## 5. Kalibrasi gate di split val (mode eval, BN tidak boleh berubah)

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def gate_logit_raw(router, f_p3, f_p2):
    # Logit gate SEBELUM kalibrasi & threshold; rumus identik dengan compute_gate_only
    z = router.squeeze_p3(f_p3) + router.squeeze_p2(f_p2)
    if hasattr(router, "_topk_spatial_pool"):
        z = router._topk_spatial_pool(z, k=getattr(router, "gate_topk", 10), alpha=getattr(router, "gate_topk_alpha", 0.7))
    else:
        z = z.mean(dim=[2, 3], keepdim=True)
    l = 5.0 * torch.tanh(router.classifier(z).view(z.shape[0], 2).float() / 5.0)
    return l[:, 1] - l[:, 0]

@torch.no_grad()
def collect_calibration_data(model, loader):
    model.eval()                                 # BN memakai running stats, TIDAK di-update
    rt = find_router(model)
    crit = model.init_criterion()                # criterion lokal: tidak menempel di model
    feats, logits, targets = {}, [], []
    h = rt.register_forward_hook(lambda m, i, o: feats.update(p3=i[0][0], p2=i[0][1]))
    try:
        for batch in loader:
            img = batch["img"].to(DEVICE).float() / 255.0
            b = {"img": img, **{k: batch[k].to(DEVICE) for k in ("cls", "bboxes", "batch_idx")}}
            det_A, det_B = model._predict_once_dual(img)   # eval -> (y, x); v8DetectionLoss membaca x
            crit.loss_A(det_A, b); crit.loss_B(det_B, b)
            la = crit.loss_A._last_per_sample_loss.cpu()
            lb = crit.loss_B._last_per_sample_loss.cpu()
            logits.append(gate_logit_raw(rt, feats["p3"], feats["p2"]).cpu())
            targets.append((la < lb).float())            # 1 = Branch A (P2) lebih baik
    finally:
        h.remove()
    return torch.cat(logits), torch.cat(targets)

def fit_temperature_scale(raw_logits, targets, n_steps=500, lr=0.05):
    T = torch.nn.Parameter(torch.tensor(1.0)); b = torch.nn.Parameter(torch.tensor(0.0))
    opt = torch.optim.LBFGS([T, b], lr=lr, max_iter=n_steps)
    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(raw_logits * T + b, targets)
        loss.backward()
        return loss
    opt.step(closure)
    with torch.no_grad():
        final = F.binary_cross_entropy_with_logits(raw_logits * T + b, targets).item()
    return T.item(), b.item(), final

calib_path = OUT / "calib.json"
if CFG["use_existing_calib"] and calib_path.is_file():
    CALIB = json.loads(calib_path.read_text())
    assert CALIB["model_fingerprint"] == FP_MODEL, "calib.json milik model/bobot lain — fit ulang."
    assert CALIB["split"] == CFG["calib_split"]
    print("Memakai calib.json yang ada:", CALIB)
else:
    raw, tgt = collect_calibration_data(model, cal_loader)
    assert raw.numel() == N_CAL, f"Logit kalibrasi {raw.numel()} != {N_CAL} gambar"
    T_cal, b_cal, bce = fit_temperature_scale(raw, tgt)
    CALIB = dict(calib_T=T_cal, calib_b=b_cal, bce=bce, split=CFG["calib_split"], n_images=int(raw.numel()),
                 target_rate=float(tgt.mean()), raw_logit_mean=float(raw.mean()), raw_logit_std=float(raw.std()),
                 activation_on_calib_split=float(((raw * T_cal + b_cal) > 0).float().mean()),
                 ckpt=CFG["router_ckpt"], ckpt_weights_key=CFG["ckpt_weights_key"],
                 model_fingerprint=FP_MODEL, code_commit=PROV["git_commit"])
    calib_path.write_text(json.dumps(CALIB, indent=1))
    print(json.dumps(CALIB, indent=1))

assert CALIB["calib_T"] > 0 or CFG.get("allow_nonpositive_T", False), ("calib_T <= 0 membalik arti gate (logit tinggi jadi 'skip P2'). "
                              "Periksa data kalibrasi; jangan dipakai.")

set_calibration(router, CALIB["calib_T"], CALIB["calib_b"])
PROV["calibration"] = CALIB
assert_model_unchanged("kalibrasi")

# Keputusan dari logit+kalibrasi HARUS sama dengan compute_gate_only (jalur deployment)
with torch.no_grad():
    for k, batch in enumerate(cal_loader):
        if k >= 2: break
        im = batch["img"].to(DEVICE).float() / 255.0
        g, p3, p2, _, _ = stage1(im)
        mine = (torch.sigmoid(gate_logit_raw(router, p3, p2) * CALIB["calib_T"] + CALIB["calib_b"]) > 0.5).float()
        assert torch.equal(mine.cpu(), g.flatten().float().cpu()), "Logit kalibrasi tidak konsisten dengan compute_gate_only"
print("✅ Logit kalibrasi konsisten dengan compute_gate_only")

## 6. Infrastruktur evaluasi akurasi (satu validator untuk SEMUA model)

In [ ]:
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.nms import non_max_suppression
from types import SimpleNamespace

class ListPredValidator(DetectionValidator):
    # Validator resmi Ultralytics (metrik, matching IoU, DetMetrics), menerima daftar
    # prediksi per-gambar (1, 4+nc, N) supaya model dengan N anchor berbeda bisa dievaluasi
    # dengan NMS & threshold yang IDENTIK.
    def postprocess(self, preds):
        out = []
        for pred in preds:
            x = non_max_suppression(pred, self.args.conf, self.args.iou, nc=0, multi_label=True,
                                    agnostic=False, max_det=self.args.max_det)[0]
            out.append({"bboxes": x[:, :4], "conf": x[:, 4], "cls": x[:, 5], "extra": x[:, 6:]})
        return out

def run_eval(name, predict_fn, loader, save_json=False):
    # predict_fn(img_float_0_1) -> (list[(1,4+nc,N)], list[dict info per gambar])
    v = ListPredValidator(args=dict(data=CFG["data_yaml"], conf=CFG["conf"], iou=CFG["iou"], max_det=CFG["max_det"],
                                    split=CFG["report_split"], imgsz=CFG["imgsz"], batch=1, half=False, plots=False,
                                    save_json=save_json, task="detect", single_cls=False, agnostic_nms=False))
    v.data, v.device = DATA, DEVICE
    v.init_metrics(SimpleNamespace(names=NAMES, end2end=False))
    records = []
    with torch.no_grad():
        for batch in loader:
            batch = v.preprocess(batch)              # /255 di sini — SATU-SATUNYA normalisasi
            preds, infos = predict_fn(batch["img"])
            v.update_metrics(v.postprocess(preds), batch)
            for f, info in zip(batch["im_file"], infos):
                records.append({"file": Path(f).name, **info})
    stats = v.get_stats()
    print(f"--- {name} ---"); v.print_results()
    assert v.seen == len(loader.dataset), f"{name}: {v.seen} gambar dievaluasi, dataset {len(loader.dataset)}"
    return dict(name=name, stats=stats, records=records, jdict=v.jdict if save_json else None, seen=v.seen)

def summarize_acc(r):
    s = r["stats"]
    return {"mAP50": s["metrics/mAP50(B)"], "mAP50-95": s["metrics/mAP50-95(B)"],
            "P": s["metrics/precision(B)"], "R": s["metrics/recall(B)"], "n_images": r["seen"]}

In [ ]:
# ===== Fungsi prediksi PyTorch =====
@torch.no_grad()
def predict_router_pt(force=None):
    def fn(im):
        preds, infos = [], []
        for i in range(im.shape[0]):                 # keputusan gate PER GAMBAR
            g, p3, p2, p4, p5 = stage1(im[i:i+1])
            gv = float(g.flatten()[0])
            logit = float(gate_logit_raw(router, p3, p2) * CALIB["calib_T"] + CALIB["calib_b"])
            use_a = (gv > 0.5) if force is None else (force == "A")
            preds.append(stage2a(p3, p2, p4, p5) if use_a else stage2b(p3, p4, p5))
            infos.append({"gate": gv, "calibrated_logit": logit, "branch": "A" if use_a else "B"})
        return preds, infos
    return fn

def load_static(path):
    from ultralytics import YOLO
    m = YOLO(path).model.float().to(DEVICE).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

@torch.no_grad()
def predict_static_pt(m):
    def fn(im):
        preds = []
        for i in range(im.shape[0]):
            out = m(im[i:i+1])
            preds.append(out[0] if isinstance(out, (tuple, list)) else out)
        return preds, [{} for _ in range(im.shape[0])]
    return fn

vanilla_pt = load_static(CFG["vanilla_ckpt"])
p2static_pt = load_static(CFG["p2static_ckpt"])

## 7. Akurasi PyTorch di split test (semua model, protokol identik)

In [ ]:
ACC = {}
R_PT = {}
for name, fn, js in (("Router (PyTorch)", predict_router_pt(None), True),
                     ("Branch A dipaksa (PyTorch)", predict_router_pt("A"), False),
                     ("Branch B dipaksa (PyTorch)", predict_router_pt("B"), False),
                     ("Vanilla (PyTorch)", predict_static_pt(vanilla_pt), True),
                     ("P2-Static (PyTorch)", predict_static_pt(p2static_pt), True)):
    R_PT[name] = run_eval(name, fn, test_loader, save_json=js and CFG["run_pycocotools"])
    ACC[name] = summarize_acc(R_PT[name])

ACT_PT = float(np.mean([r["branch"] == "A" for r in R_PT["Router (PyTorch)"]["records"]]))
ACC["Router (PyTorch)"]["activation"] = ACT_PT
print(f"\nAktivasi P2 (PyTorch, {CFG['report_split']}): {ACT_PT*100:.2f}%")
assert_model_unchanged("evaluasi akurasi PyTorch")

## 8. Ekspor ONNX dari model yang SAMA (salinan CPU), lalu verifikasi file yang sama persis

In [ ]:
class _Stage1Out(nn.Module):
    def __init__(self, s): super().__init__(); self.s = s
    def forward(self, x): return self.s(x)

class _StaticOut(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        out = self.m(x)
        return out[0] if isinstance(out, (tuple, list)) else out

def export_onnx(module, args, path, in_names, out_names):
    # Eksporter & opset yang SAMA untuk kelima model, supaya graf TensorRT sebanding
    with torch.no_grad():
        torch.onnx.export(module.eval(), args, str(path), input_names=in_names, output_names=out_names,
                          opset_version=17, dynamic_axes=None, dynamo=False)

TAG = FP_MODEL[:8]
ONNX = {k: OUT / f"{k}_{TAG}.onnx" for k in ("stage1", "stage2a", "stage2b")}
ONNX["vanilla"] = OUT / "vanilla.onnx"; ONNX["p2static"] = OUT / "p2static.onnx"

m_cpu = copy.deepcopy(model).cpu().eval()           # salinan: model evaluasi tetap di DEVICE dan utuh
assert get_calibration(find_router(m_cpu)) == get_calibration(router), "Kalibrasi tidak ikut tersalin"
s1c, s2ac, s2bc = Stage1_BackboneNeckGate(m_cpu).eval(), Stage2A_WithP2(m_cpu).eval(), Stage2B_NoP2(m_cpu).eval()
x0 = next(iter(test_loader))["img"].float() / 255.0
with torch.no_grad():
    _, p3, p2, p4, p5 = s1c(x0)
export_onnx(_Stage1Out(s1c), (x0,), ONNX["stage1"], ["image"], ["gate", "p3_neck", "p2_backbone", "p4_neck", "p5_sppf"])
export_onnx(s2ac, (p3, p2, p4, p5), ONNX["stage2a"], ["p3_neck", "p2_backbone", "p4_neck", "p5_sppf"], ["y_A"])
export_onnx(s2bc, (p3, p4, p5), ONNX["stage2b"], ["p3_neck", "p4_neck", "p5_sppf"], ["y_B"])
export_onnx(_StaticOut(copy.deepcopy(vanilla_pt).cpu()), (x0,), ONNX["vanilla"], ["image"], ["y"])
export_onnx(_StaticOut(copy.deepcopy(p2static_pt).cpu()), (x0,), ONNX["p2static"], ["image"], ["y"])
PROV["onnx_sha256"] = {k: sha256_file(p) for k, p in ONNX.items()}
assert_model_unchanged("ekspor ONNX")
print("✅ ONNX:", {k: p.name for k, p in ONNX.items()})

In [ ]:
import onnxruntime as ort

def ort_run(path, feed):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    names = [i.name for i in sess.get_inputs()]
    return sess.run(None, {n: feed[n] for n in names})

def rel_err(a, b):
    return float(np.abs(a - b).max() / (np.abs(b).max() + 1e-12))

# Verifikasi pada gambar ASLI, memakai file ONNX yang SAMA dengan yang akan dibangun jadi engine
worst = {}
with torch.no_grad():
    for k, batch in enumerate(test_loader):
        if k >= CFG["n_onnx_check"]: break
        x = batch["img"].float() / 255.0
        g, p3, p2, p4, p5 = s1c(x)
        o1 = ort_run(ONNX["stage1"], {"image": x.numpy()})
        assert float(o1[0].flatten()[0]) == float(g.flatten()[0]), f"Gate ONNX != PyTorch pada {batch['im_file'][0]}"
        worst["stage1_feats"] = max(worst.get("stage1_feats", 0), *[rel_err(a, b.numpy()) for a, b in zip(o1[1:], (p3, p2, p4, p5))])
        yA = ort_run(ONNX["stage2a"], {"p3_neck": p3.numpy(), "p2_backbone": p2.numpy(), "p4_neck": p4.numpy(), "p5_sppf": p5.numpy()})[0]
        yB = ort_run(ONNX["stage2b"], {"p3_neck": p3.numpy(), "p4_neck": p4.numpy(), "p5_sppf": p5.numpy()})[0]
        worst["stage2a"] = max(worst.get("stage2a", 0), rel_err(yA, s2ac(p3, p2, p4, p5).numpy()))
        worst["stage2b"] = max(worst.get("stage2b", 0), rel_err(yB, s2bc(p3, p4, p5).numpy()))
        for nm, mdl in (("vanilla", vanilla_pt), ("p2static", p2static_pt)):
            ref = _StaticOut(copy.deepcopy(mdl).cpu())(x).numpy()
            worst[nm] = max(worst.get(nm, 0), rel_err(ort_run(ONNX[nm], {"image": x.numpy()})[0], ref))
print("Error relatif maksimum ONNX vs PyTorch:", {k: f"{v:.2e}" for k, v in worst.items()})
assert all(v < 1e-4 for v in worst.values()), "Selisih ONNX vs PyTorch melebihi 1e-4 (relatif)"
PROV["onnx_check_max_rel_err"] = worst

## 9. Build engine TensorRT FP32 (satu fungsi untuk kelima model)

In [ ]:
try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit  # noqa: F401  — buat context CUDA sebelum cuda.Stream()
    HAS_TRT = torch.cuda.is_available()
except Exception as e:
    HAS_TRT = False
    print("TensorRT/pycuda tidak tersedia:", e)

ENGINE = {k: p.with_suffix(".engine") for k, p in ONNX.items()}
if HAS_TRT:
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    def build_engine(onnx_path, engine_path):
        builder = trt.Builder(TRT_LOGGER)
        network = builder.create_network()
        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        # Tanpa flag presisi = FP32 untuk semua model
        engine_bytes = builder.build_serialized_network(network, config)
        assert engine_bytes is not None, f"Gagal build {onnx_path}"
        with open(engine_path, "wb") as f:
            f.write(engine_bytes)
    for k in ONNX:
        build_engine(ONNX[k], ENGINE[k])
    PROV["engine_sha256"] = {k: sha256_file(p) for k, p in ENGINE.items()}
    PROV["versions"]["tensorrt"] = trt.__version__
    print("✅ Engine:", {k: p.name for k, p in ENGINE.items()})

In [ ]:
if HAS_TRT:
    class TRTEngineOptimized:
        # Buffer device/host dialokasikan SEKALI. Dipakai Vanilla & P2-Static.
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(TRT_LOGGER) as rt:
                self.engine = rt.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context()
            self.stream = cuda.Stream()
            self.inp, self.out, self.d, self.h = [], [], {}, {}
            for i in range(self.engine.num_io_tensors):
                n = self.engine.get_tensor_name(i)
                (self.inp if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT else self.out).append(n)
            for n in self.inp:
                shape, dt = tuple(self.engine.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n))
                self.ctx.set_input_shape(n, shape)
                self.d[n] = cuda.mem_alloc(int(np.prod(shape)) * np.dtype(dt).itemsize)
                self.ctx.set_tensor_address(n, int(self.d[n]))
            for n in self.out:
                self.h[n] = cuda.pagelocked_empty(tuple(self.ctx.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n)))
                self.d[n] = cuda.mem_alloc(self.h[n].nbytes)
                self.ctx.set_tensor_address(n, int(self.d[n]))
        def infer(self, image_np):
            cuda.memcpy_htod_async(self.d[self.inp[0]], image_np, self.stream)   # H2D gambar
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            for n in self.out:
                cuda.memcpy_dtoh_async(self.h[n], self.d[n], self.stream)          # D2H output
            self.stream.synchronize()
            return self.h[self.out[0]]

    class TRTEngineChainable:
        # Router: Stage1 menerima gambar dari host; Stage2 menerima pointer device dari Stage1.
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(TRT_LOGGER) as rt:
                self.engine = rt.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context()
            self.stream = cuda.Stream()
            self.inp, self.out, self.d_in, self.d_out, self.h_out = [], [], {}, {}, {}
            for i in range(self.engine.num_io_tensors):
                n = self.engine.get_tensor_name(i)
                (self.inp if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT else self.out).append(n)
            for n in self.inp:
                shape, dt = tuple(self.engine.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n))
                self.ctx.set_input_shape(n, shape)
                self.d_in[n] = cuda.mem_alloc(int(np.prod(shape)) * np.dtype(dt).itemsize)
            for n in self.out:
                self.h_out[n] = cuda.pagelocked_empty(tuple(self.ctx.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n)))
                self.d_out[n] = cuda.mem_alloc(self.h_out[n].nbytes)
                self.ctx.set_tensor_address(n, int(self.d_out[n]))
        def run_from_host(self, image_np):
            n = self.inp[0]
            cuda.memcpy_htod_async(self.d_in[n], image_np, self.stream)
            self.ctx.set_tensor_address(n, int(self.d_in[n]))
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
        def read_scalar(self, name):
            cuda.memcpy_dtoh_async(self.h_out[name], self.d_out[name], self.stream)   # satu-satunya D2H antar-stage
            self.stream.synchronize()
            return float(self.h_out[name].flatten()[0])
        def run_from_device(self, ptrs):
            for n, p in ptrs.items():
                self.ctx.set_tensor_address(n, int(p))
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            for n in self.out:
                cuda.memcpy_dtoh_async(self.h_out[n], self.d_out[n], self.stream)   # D2H output akhir
            self.stream.synchronize()
            return self.h_out[self.out[0]]

    s1_trt, s2a_trt, s2b_trt = (TRTEngineChainable(ENGINE[k]) for k in ("stage1", "stage2a", "stage2b"))
    vanilla_trt, p2static_trt = TRTEngineOptimized(ENGINE["vanilla"]), TRTEngineOptimized(ENGINE["p2static"])
    A_KEYS, B_KEYS = ("p3_neck", "p2_backbone", "p4_neck", "p5_sppf"), ("p3_neck", "p4_neck", "p5_sppf")

    def router_trt(image_np, force=None):
        s1_trt.run_from_host(image_np)
        g = s1_trt.read_scalar("gate")
        use_a = (g > 0.5) if force is None else (force == "A")
        if use_a:
            y = s2a_trt.run_from_device({k: s1_trt.d_out[k] for k in A_KEYS})
        else:
            y = s2b_trt.run_from_device({k: s1_trt.d_out[k] for k in B_KEYS})
        return y, g, "A" if use_a else "B"

## 10. Akurasi dari output TensorRT + konsistensi gate TRT vs PyTorch untuk SEMUA gambar

In [ ]:
if HAS_TRT:
    def predict_router_trt(im):
        preds, infos = [], []
        for i in range(im.shape[0]):
            y, g, br = router_trt(np.ascontiguousarray(im[i:i+1].cpu().numpy(), dtype=np.float32))
            preds.append(torch.from_numpy(y.copy()).to(im.device))
            infos.append({"gate": g, "branch": br})
        return preds, infos

    def predict_static_trt(eng):
        def fn(im):
            preds = [torch.from_numpy(eng.infer(np.ascontiguousarray(im[i:i+1].cpu().numpy(), dtype=np.float32)).copy()).to(im.device)
                     for i in range(im.shape[0])]
            return preds, [{} for _ in range(im.shape[0])]
        return fn

    R_TRT = {}
    for name, fn in (("Router (TensorRT)", predict_router_trt),
                     ("Vanilla (TensorRT)", predict_static_trt(vanilla_trt)),
                     ("P2-Static (TensorRT)", predict_static_trt(p2static_trt))):
        R_TRT[name] = run_eval(name, fn, test_loader, save_json=CFG["run_pycocotools"])
        ACC[name] = summarize_acc(R_TRT[name])

    pt_rec, trt_rec = R_PT["Router (PyTorch)"]["records"], R_TRT["Router (TensorRT)"]["records"]
    assert [r["file"] for r in pt_rec] == [r["file"] for r in trt_rec], "Urutan gambar berbeda"
    mism = [(a["file"], a["branch"], b["branch"], a["calibrated_logit"]) for a, b in zip(pt_rec, trt_rec) if a["branch"] != b["branch"]]
    # Perbedaan HANYA ditoleransi untuk gambar yang logit terkalibrasinya praktis tepat di batas (|logit| < 1e-3),
    # yaitu selisih numerik FP32 PyTorch vs TensorRT; selain itu dianggap bug.
    serious = [m for m in mism if abs(m[3]) >= 1e-3]
    ACT_TRT = float(np.mean([r["branch"] == "A" for r in trt_rec]))
    ACC["Router (TensorRT)"]["activation"] = ACT_TRT
    print(f"Keputusan gate TRT vs PyTorch: {len(pt_rec) - len(mism)}/{len(pt_rec)} cocok | aktivasi PT={ACT_PT*100:.2f}% TRT={ACT_TRT*100:.2f}%")
    for nm in ("Router", "Vanilla", "P2-Static"):
        print(f"  ΔmAP50 TRT−PT {nm}: {ACC[nm + ' (TensorRT)']['mAP50'] - ACC[nm + ' (PyTorch)']['mAP50']:+.4f}")
    if mism: print("Beda keputusan (file, PT, TRT, logit terkalibrasi PT):", mism[:10])
    assert not serious, f"❌ {len(serious)} keputusan gate berbeda jauh dari batas, contoh: {serious[:5]}. Hentikan dan diagnosis."
    PROV["gate_match_trt_vs_pt"] = f"{len(pt_rec) - len(mism)}/{len(pt_rec)} (beda di batas: {len(mism)})"

## 11. Latensi — adil: engine pre-alloc, gambar test asli yang sama, diukur bergantian

In [ ]:
def lat_stats(a):
    a = np.asarray(a)
    return dict(mean=a.mean(), std=a.std(), p50=np.percentile(a, 50), p95=np.percentile(a, 95),
                p99=np.percentile(a, 99), min=a.min(), max=a.max(), n=int(a.size))

LAT = {}
if HAS_TRT:
    def as_np(batch):     # konversi DI LUAR timer, identik untuk semua model
        return np.ascontiguousarray(batch["img"].numpy().astype(np.float32) / 255.0)

    # Warm-up: SEMUA engine, termasuk Stage2A DAN Stage2B
    for k, batch in enumerate(test_loader):
        if k >= CFG["n_warmup"]: break
        im = as_np(batch)
        vanilla_trt.infer(im); p2static_trt.infer(im)
        router_trt(im, force="A"); router_trt(im, force="B")

    t = {"Vanilla (TensorRT)": [], "P2-Static (TensorRT)": [], "Router (TensorRT)": []}
    br_lat = {"A": [], "B": []}
    bench_branch = []
    for rep in range(CFG["n_reps"]):
        for batch in test_loader:
            im = as_np(batch)
            order = list(t.keys())
            if rep % 2: order = order[::-1]                     # urutan dibalik tiap putaran
            for name in order:
                if name.startswith("Router"):
                    t0 = time.perf_counter(); _, _, br = router_trt(im); dt = (time.perf_counter() - t0) * 1e3
                    br_lat[br].append(dt)
                    if rep == 0: bench_branch.append(br)
                elif name.startswith("Vanilla"):
                    t0 = time.perf_counter(); vanilla_trt.infer(im); dt = (time.perf_counter() - t0) * 1e3
                else:
                    t0 = time.perf_counter(); p2static_trt.infer(im); dt = (time.perf_counter() - t0) * 1e3
                t[name].append(dt)

    for name, arr in t.items():
        LAT[name] = lat_stats(arr)
    LAT["Router (TensorRT) — saat Branch A"] = lat_stats(br_lat["A"]) if br_lat["A"] else None
    LAT["Router (TensorRT) — saat Branch B"] = lat_stats(br_lat["B"]) if br_lat["B"] else None
    ACT_BENCH = float(np.mean([b == "A" for b in bench_branch]))
    print(f"Aktivasi saat benchmark: {ACT_BENCH*100:.2f}% | evaluasi TRT {ACT_TRT*100:.2f}% | evaluasi PyTorch {ACT_PT*100:.2f}%")
    assert abs(ACT_BENCH - ACT_TRT) < 1e-9, "❌ Aktivasi benchmark != aktivasi saat evaluasi akurasi TensorRT"

# Referensi PyTorch eager (BUKAN pembanding routing): model dual-branch utuh, FP32, diukur di sesi ini
if torch.cuda.is_available():
    ref = []
    with torch.no_grad():
        for k, batch in enumerate(test_loader):
            im = batch["img"].to(DEVICE).float() / 255.0
            torch.cuda.synchronize(); t0 = time.perf_counter()
            model._predict_once_dual(im)
            torch.cuda.synchronize(); dt = (time.perf_counter() - t0) * 1e3
            if k >= CFG["n_warmup"]: ref.append(dt)
    LAT["Dual-branch PyTorch eager (referensi)"] = lat_stats(ref)

for name, s in LAT.items():
    if s: print(f"{name:42s} mean={s['mean']:.3f} std={s['std']:.3f} p50={s['p50']:.3f} p95={s['p95']:.3f} p99={s['p99']:.3f} n={s['n']}")
assert_model_unchanged("benchmark latency")

## 12. Tabel ringkasan + penyimpanan hasil

In [ ]:
import pandas as pd
rows = []
for name, a in ACC.items():
    lat = LAT.get(name)
    rows.append({"model": name, **{k: a.get(k) for k in ("mAP50", "mAP50-95", "P", "R", "n_images")},
                 "aktivasi_P2": a.get("activation"),
                 **({f"lat_{k}_ms": lat[k] for k in ("mean", "std", "p50", "p95", "p99")} if lat else {})})
TABLE = pd.DataFrame(rows)
TABLE["metode_mAP"] = "Ultralytics DetMetrics; letterbox persegi 640; conf 0.001; IoU NMS 0.7; batch 1"
pd.set_option("display.width", 250)
print(TABLE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

TABLE.to_csv(OUT / "hasil_akurasi_latency.csv", index=False)
(OUT / "latency.json").write_text(json.dumps(LAT, indent=1, default=float))
pd.DataFrame(R_PT["Router (PyTorch)"]["records"]).to_csv(OUT / "keputusan_gate_test.csv", index=False)
(OUT / "provenance.json").write_text(json.dumps({**PROV, "cfg": CFG}, indent=1, default=str))
print("✅ Disimpan di", OUT)

## 13. (Opsional) Breakdown pycocotools — mAP_small/medium/large, AR

In [ ]:
if CFG["run_pycocotools"]:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    # Ground truth dari label dataset test, koordinat piksel RESOLUSI ASLI.
    # category_id = cls + 1 dan image_id = int(stem): SAMA dengan pred_to_json Ultralytics (class_map = 1..nc).
    ds = test_loader.dataset
    images, anns, aid = [], [], 1
    for lab in ds.labels:
        stem = Path(lab["im_file"]).stem
        iid = int(stem) if stem.isnumeric() else stem
        h, w = lab["shape"]
        images.append({"id": iid, "file_name": Path(lab["im_file"]).name, "height": int(h), "width": int(w)})
        for c, (cx, cy, bw, bh) in zip(lab["cls"].reshape(-1), lab["bboxes"].reshape(-1, 4)):
            bw_px, bh_px = float(bw * w), float(bh * h)
            anns.append({"id": aid, "image_id": iid, "category_id": int(c) + 1,
                         "bbox": [float(cx * w - bw_px / 2), float(cy * h - bh_px / 2), bw_px, bh_px],
                         "area": bw_px * bh_px, "iscrowd": 0})
            aid += 1
    gt_json = {"images": images, "annotations": anns, "categories": [{"id": i + 1, "name": str(n)} for i, n in NAMES.items()]}
    (OUT / "gt_test_coco.json").write_text(json.dumps(gt_json))
    coco_gt = COCO(str(OUT / "gt_test_coco.json"))
    assert len(coco_gt.getImgIds()) == N_TEST

    COCO_STATS = {}
    keys = ["mAP50-95", "mAP50", "mAP75", "mAP_small", "mAP_medium", "mAP_large", "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]
    for name, r in {**R_PT, **globals().get("R_TRT", {})}.items():
        if not r.get("jdict"):
            continue
        ids_pred = {d["image_id"] for d in r["jdict"]}
        assert ids_pred <= set(coco_gt.getImgIds()), f"{name}: image_id prediksi tidak ada di GT"
        dt = coco_gt.loadRes(r["jdict"])
        E = COCOeval(coco_gt, dt, "bbox"); E.evaluate(); E.accumulate(); E.summarize()
        COCO_STATS[name] = dict(zip(keys, map(float, E.stats)))
    COCO_DF = pd.DataFrame(COCO_STATS).T
    COCO_DF["metode"] = "pycocotools (resolusi asli, maxDets 100) — JANGAN dicampur dengan kolom Ultralytics"
    print(COCO_DF.to_string(float_format=lambda v: f"{v:.4f}"))
    COCO_DF.to_csv(OUT / "hasil_pycocotools.csv")

## 14. (Opsional) Analisis perilaku gate di split test — model bersih, kalibrasi dari val

In [ ]:
if CFG["run_gate_analysis"]:
    from scipy import stats as st
    crit = model.init_criterion()
    rows = []
    with torch.no_grad():
        for batch, lab in zip(test_loader, test_loader.dataset.labels):
            im = batch["img"].to(DEVICE).float() / 255.0
            g, p3, p2, p4, p5 = stage1(im)
            prob = float(torch.sigmoid(gate_logit_raw(router, p3, p2) * CALIB["calib_T"] + CALIB["calib_b"]))
            det_A, det_B = model._predict_once_dual(im)
            b = {"img": im, **{k: batch[k].to(DEVICE) for k in ("cls", "bboxes", "batch_idx")}}
            crit.loss_A(det_A, b); crit.loss_B(det_B, b)
            la, lb = float(crit.loss_A._last_per_sample_loss[0]), float(crit.loss_B._last_per_sample_loss[0])
            h, w = lab["shape"]
            bx = lab["bboxes"].reshape(-1, 4)
            area_px = bx[:, 2] * w * bx[:, 3] * h                                   # resolusi asli
            area_lb = (batch["bboxes"][:, 2] * batch["bboxes"][:, 3]).numpy()       # ternormalisasi letterbox (definisi lama)
            rows.append({"file": Path(lab["im_file"]).name, "gate_prob": prob, "branch": "A" if float(g.flatten()[0]) > 0.5 else "B",
                         "loss_A": la, "loss_B": lb, "loss_diff_B_minus_A": lb - la,
                         "n_objects": int(len(bx)), "n_small_coco": int((area_px < 32 ** 2).sum()),
                         "small_ratio_coco": float((area_px < 32 ** 2).mean()) if len(bx) else np.nan,
                         "n_small_old_def": int((area_lb < 0.01).sum()),
                         "mean_area_px": float(area_px.mean()) if len(bx) else np.nan})
    GA = pd.DataFrame(rows)
    GA.to_csv(OUT / "analisis_gate_test.csv", index=False)
    assert_model_unchanged("analisis gate")

    def sp(x, y):
        m = GA[[x, y]].dropna()
        r, p = st.spearmanr(m[x], m[y]); return f"r={r:+.3f}, p={p:.2e}, n={len(m)}"
    print("gate_prob vs n_objects            :", sp("gate_prob", "n_objects"))
    print("gate_prob vs small_ratio_coco     :", sp("gate_prob", "small_ratio_coco"))
    print("gate_prob vs n_small_coco         :", sp("gate_prob", "n_small_coco"))
    print("gate_prob vs n_small_old_def      :", sp("gate_prob", "n_small_old_def"), "(definisi lama, hanya pembanding)")
    print("gate_prob vs mean_area_px         :", sp("gate_prob", "mean_area_px"))
    print("loss_diff(B−A) vs n_objects       :", sp("loss_diff_B_minus_A", "n_objects"))
    m = GA[["gate_prob", "small_ratio_coco", "n_objects"]].dropna()
    if min(m[c].nunique() for c in m.columns) > 1:
        rx = st.rankdata(m["gate_prob"]); ry = st.rankdata(m["small_ratio_coco"]); rz = st.rankdata(m["n_objects"])
        res = lambda a, b: a - np.polyval(np.polyfit(b, a, 1), b)
        r_part, p_part = st.pearsonr(res(rx, rz), res(ry, rz))
        print(f"Parsial (Spearman) gate vs small_ratio | n_objects: r={r_part:+.3f}, p={p_part:.2e}")
    else:
        print("Parsial: dilewati (salah satu variabel konstan)")
    amb = GA["gate_prob"].between(0.4, 0.6)
    u = st.mannwhitneyu(GA.loc[amb, "loss_diff_B_minus_A"].abs(), GA.loc[~amb, "loss_diff_B_minus_A"].abs(), alternative="less")
    print(f"|loss_diff| ambigu (n={amb.sum()}) < tegas (n={(~amb).sum()}): Mann-Whitney p={u.pvalue:.3f}")
    print(GA["gate_prob"].describe())